# BiLSTM + Electra-base (Weighted Ensemble)

This notebook combines the two previously trained models — the from-scratch BiLSTM (with attention) and the fine-tuned Electra-base — into a single weighted ensemble for final MCQ answer prediction.

**Approach:** rather than retraining, this notebook loads the best saved checkpoints from each individual model's notebook and runs inference with both. Their softmax probability outputs are combined via a fixed weighted average (Electra weighted more heavily, given its stronger individual performance), and the top-3 combined predictions are taken as the final answer ranking.

# Importing Libraries

In [1]:
import os
import re
import random
import zipfile
import numpy as np
import pandas as pd
from collections import Counter
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
import wandb
from kaggle_secrets import UserSecretsClient
from dataclasses import dataclass, asdict

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

sns.set_theme(style="whitegrid", palette="muted")

Using device: cuda


# Loading Dataset

In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
print("Datasets loaded!")

Datasets loaded!


# Define the Model

## Configuration

In [3]:
@dataclass
class BiLSTMConfig:
    model_name: str = "bilstm-simple-attention-1"
    vocab_size: int = 25000
    embed_dim: int = 128
    hidden_dim: int = 256
    max_seq_len: int = 200
    
    epochs: int = 21
    batch_size: int = 32  
    learning_rate: float = 2e-3 
    weight_decay: float = 0.05

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    wandb_project: str = "23f2004791-t22026"
    
    def to_dict(self):
        return asdict(self)

bilstm_cfg = BiLSTMConfig()

@dataclass
class ElectraConfig:
    model_name: str = "google/electra-base-discriminator"
    max_seq_len: int = 150
    
    epochs: int = 5
    batch_size: int = 16
    gradient_accumulation_steps: int = 1
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    wandb_project: str = "23f2004791-t22026"
    
    def to_dict(self):
        return asdict(self)

elec_cfg = ElectraConfig()

In [4]:
try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_api_key)
    print("Successfully logged into Weights & Biases!")
except Exception as e:
    print(f"W&B Login Failed.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adrija935 (23f2004791-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged into Weights & Biases!


## Tokenizer and Vocabulary Builder

### BiLSTM

In [5]:
class SimpleVocab:
    def __init__(self, min_freq=2):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1} 
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.min_freq = min_freq
        self.vocab_size = 2

    def tokenize(self, text):
        # Lowercase and remove punctuation
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]', '', text)
        return text.split()

    def build_vocab(self, texts):
        word_counts = Counter()
        for text in texts:
            word_counts.update(self.tokenize(text))
            
        for word, count in word_counts.items():
            if count >= self.min_freq:
                self.word2idx[word] = self.vocab_size
                self.idx2word[self.vocab_size] = word
                self.vocab_size += 1

    def encode(self, text, max_len=128):
        tokens = self.tokenize(text)
        ids = [self.word2idx.get(word, 1) for word in tokens] 
        if len(ids) > max_len:
            ids = ids[:max_len] # Clipping
        else:
            ids = ids + [0] * (max_len - len(ids)) # Padding
        return ids


# Collect all text to build vocabulary
corpus = train_df['prompt'].tolist()
for col in ['A', 'B', 'C', 'D', 'E']:
    corpus.extend(train_df[col].tolist())
    
vocab = SimpleVocab(min_freq=2)
vocab.build_vocab(corpus)
bilstm_cfg.vocab_size = vocab.vocab_size
print(f"Vocabulary size built: {vocab.vocab_size} unique tokens.")

Vocabulary size built: 3081 unique tokens.


### Electra

In [6]:
tokenizer = AutoTokenizer.from_pretrained(elec_cfg.model_name)

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## PyTorch Dataset and DataLoaders

### BiLSTM

In [7]:
class MCQBiLSTMDataset(Dataset): 
    def __init__(self, df, vocab, max_len=128, is_test=False):
        self.df = df
        self.vocab = vocab 
        self.max_len = max_len
        self.is_test = is_test
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        options = [str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
        
        input_ids = []
        for opt in options:
            # Combine prompt and option text 
            text = prompt + " " + opt
            ids = self.vocab.encode(text, max_len=self.max_len)
            input_ids.append(ids)
            
        input_tensor = torch.tensor(input_ids, dtype=torch.long) 
        if not self.is_test:
            label = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
            return input_tensor, label
        return input_tensor

### Electra

In [8]:
class MCQElectraDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        options = [str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
        
        # Duplicate prompt 5 times to pair with each option
        first_sentences = [prompt] * 5
        second_sentences = options

        # Tokenize (Prompt, Option) pairs
        encoding = self.tokenizer(
            first_sentences,
            second_sentences,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        
        item = {
            'input_ids': encoding['input_ids'],
            'attention_mask': encoding['attention_mask']
        }
        
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids']
            
        if not self.is_test:
            item['label'] = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
            
        return item

## Train-test Split

In [9]:
# train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=SEED)

# # BiLSTM
# bilstm_train_dataset = MCQBiLSTMDataset(train_split, vocab, max_len=bilstm_cfg.max_seq_len)
# bilstm_val_dataset = MCQBiLSTMDataset(val_split, vocab, max_len=bilstm_cfg.max_seq_len)
bilstm_test_dataset = MCQBiLSTMDataset(test_df, vocab, max_len=bilstm_cfg.max_seq_len, is_test=True)

# bilstm_train_loader = DataLoader(bilstm_train_dataset, batch_size=bilstm_cfg.batch_size, shuffle=True)
# bilstm_val_loader = DataLoader(bilstm_val_dataset, batch_size=bilstm_cfg.batch_size, shuffle=False)
bilstm_test_loader = DataLoader(bilstm_test_dataset, batch_size=bilstm_cfg.batch_size, shuffle=False)

# # Electra
# elec_train_dataset = MCQElectraDataset(train_split, tokenizer, max_len=elec_cfg.max_seq_len)
# elec_val_dataset = MCQElectraDataset(val_split, tokenizer, max_len=elec_cfg.max_seq_len)
elec_test_dataset = MCQElectraDataset(test_df, tokenizer, max_len=elec_cfg.max_seq_len, is_test=True)

# elec_train_loader = DataLoader(elec_train_dataset, batch_size=elec_cfg.batch_size, shuffle=True)
# elec_val_loader = DataLoader(elec_val_dataset, batch_size=elec_cfg.batch_size, shuffle=False)
elec_test_loader = DataLoader(elec_test_dataset, batch_size=elec_cfg.batch_size, shuffle=False)

## Model Architecture

In [10]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim * 2, 1)

    def forward(self, lstm_out):
        scores = self.attention(lstm_out)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(weights * lstm_out, dim=1) 
        return context

class MCQBiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, dropout=0.3):
        super(MCQBiLSTMAttention, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        
        self.attention = Attention(hidden_dim)
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        batch_size, num_options, max_len = x.shape
        x = x.view(batch_size * num_options, max_len) 
        embedded = self.embedding(x) 
        lstm_out, _ = self.lstm(embedded)
        attended_vector = self.attention(lstm_out) 
        scores = self.fc(attended_vector) 
        logits = scores.view(batch_size, num_options) 
        return logits

In [11]:
bilstm_model = MCQBiLSTMAttention(vocab_size=vocab.vocab_size,hidden_dim=bilstm_cfg.hidden_dim,embed_dim=bilstm_cfg.embed_dim).to(bilstm_cfg.device)

# criterion = nn.CrossEntropyLoss()
# bilstm_optimizer = torch.optim.AdamW(bilstm_model.parameters(), lr=bilstm_cfg.learning_rate, weight_decay=bilstm_cfg.weight_decay)
# bilstm_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(bilstm_optimizer, T_max=bilstm_cfg.epochs, eta_min=1e-5)


electra_model = AutoModelForMultipleChoice.from_pretrained(elec_cfg.model_name).to(elec_cfg.device)

# total_steps = (len(elec_train_loader) // elec_cfg.gradient_accumulation_steps) * elec_cfg.epochs
# warmup_steps = int(total_steps * elec_cfg.warmup_ratio)
# elec_optimizer = torch.optim.AdamW(electra_model.parameters(), lr=elec_cfg.learning_rate, weight_decay=elec_cfg.weight_decay)
# elec_scheduler = get_cosine_schedule_with_warmup(elec_optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForMultipleChoice LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
electra.embeddings_project.bias                   | UNEXPECTED | 
classifier.bias                                   | MISSING    | 
sequence_summary.summary.weight                   | MISSING    | 
classifier.weight                                 | MISSING    | 
sequence_summary.summary.bias                     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

## Scoring Metric (MAP@3)

In [12]:
# MAP@3 Metric
def compute_map_at_3(predictions, targets):
    scores = []
    for top_preds, target in zip(predictions, targets):
        score = 0.0
        for rank, pred in enumerate(top_preds):
            if pred == target:
                score = 1.0 / (rank + 1)
                break
        scores.append(score)
    return np.mean(scores)

# Training and Validation

In [13]:
# wandb.init(
#     project=bilstm_cfg.wandb_project, 
#     name="ensemble-exp-4"
# )
# wandb.define_metric("bilstm/*", step_metric="bilstm/epoch")
# wandb.define_metric("electra/*", step_metric="electra/epoch")

# # BiLSTM training and validation
# best_map3 = 0.0

# for epoch in range(bilstm_cfg.epochs):
#     bilstm_model.train()
#     running_loss = 0.0
#     loop = tqdm(bilstm_train_loader, desc=f"Epoch {epoch+1}/{bilstm_cfg.epochs}")
#     for inputs, labels in loop:
#         inputs, labels = inputs.to(bilstm_cfg.device), labels.to(bilstm_cfg.device)
        
#         bilstm_optimizer.zero_grad()
#         outputs = bilstm_model(inputs)
#         loss = criterion(outputs, labels)
#         loss.backward()
        
#         # Gradient clipping prevents gradient saturation/explosion
#         torch.nn.utils.clip_grad_norm_(bilstm_model.parameters(), max_norm=1.0)
        
#         bilstm_optimizer.step()
#         running_loss += loss.item() * inputs.size(0)
        
#     bilstm_scheduler.step()
#     train_loss = running_loss / len(bilstm_train_loader.dataset)
    
#     # Validation Loop
#     bilstm_model.eval()
#     val_loss = 0.0
#     all_top3_preds = []
#     all_targets = []
    
#     with torch.no_grad():
#         val_loop = tqdm(bilstm_val_loader)
#         for inputs, labels in val_loop:
#             inputs, labels = inputs.to(bilstm_cfg.device), labels.to(bilstm_cfg.device)
#             outputs = bilstm_model(inputs)
#             loss = criterion(outputs, labels)
            
#             val_loss += loss.item() * inputs.size(0)
            
#             _, top3_indices = torch.topk(outputs, k=3, dim=1)
#             all_top3_preds.append(top3_indices.cpu().numpy())
#             all_targets.append(labels.cpu().numpy())
            
#     val_loss = val_loss / len(bilstm_val_loader.dataset)
#     all_top3_preds = np.concatenate(all_top3_preds, axis=0)
#     all_targets = np.concatenate(all_targets, axis=0)
    
#     val_map3 = compute_map_at_3(all_top3_preds, all_targets)
        
#     wandb.log({
#         "bilstm/epoch": epoch + 1,
#         "bilstm/train_loss": train_loss,
#         "bilstm/val_loss": val_loss,
#         "bilstm/val_map3": val_map3,
#         "bilstm/best_val_map3": best_map3,
#         "bilstm/lr": bilstm_scheduler.get_last_lr()[0]
#     })
    
#     print(f"Epoch {epoch+1:02d}/{bilstm_cfg.epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val MAP@3: {val_map3:.4f}")

#     if val_map3 > best_map3:
#         best_map3 = val_map3
#         torch.save(bilstm_model.state_dict(), "best_bilstm_attn_model.pt")
#         print(f"New best bilstm model saved with MAP@3: {best_map3:.4f}")
# torch.save(bilstm_model.state_dict(), "best_bilstm_attn_model.pt")

# # Electra-base training and validation

# best_map3 = 0.0

# scaler = torch.amp.GradScaler('cuda')

# for epoch in range(elec_cfg.epochs):
#     electra_model.train()
#     running_loss = 0.0
#     elec_optimizer.zero_grad()
    
#     loop = tqdm(elec_train_loader, desc=f"Epoch {epoch+1}/{elec_cfg.epochs}")
#     for step, batch in enumerate(loop):
#         input_ids = batch['input_ids'].to(elec_cfg.device)
#         attention_mask = batch['attention_mask'].to(elec_cfg.device)
#         labels = batch['label'].to(elec_cfg.device)
        
#         kwargs = {
#             'input_ids': input_ids,
#             'attention_mask': attention_mask,
#             'labels': labels
#         }
#         if 'token_type_ids' in batch:
#             kwargs['token_type_ids'] = batch['token_type_ids'].to(elec_cfg.device)
            
#         with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
#             outputs = electra_model(**kwargs)
#             loss = outputs.loss / elec_cfg.gradient_accumulation_steps
            
#         scaler.scale(loss).backward()
        
#         if (step + 1) % elec_cfg.gradient_accumulation_steps == 0 or (step + 1) == len(elec_train_loader):
#             scaler.unscale_(elec_optimizer)
#             torch.nn.utils.clip_grad_norm_(electra_model.parameters(), max_norm=1.0)

#             scaler.step(elec_optimizer)
#             scaler.update()
            
#             elec_scheduler.step()
#             elec_optimizer.zero_grad()
            
#         running_loss += outputs.loss.item() * input_ids.size(0)
        
#     train_loss = running_loss / len(elec_train_loader.dataset)
    
#     # Validation Loop
#     electra_model.eval()
#     val_loss = 0.0
#     all_top3_preds = []
#     all_top1_preds = []
#     all_targets = []
    
#     with torch.no_grad():
#         val_loop = tqdm(elec_val_loader, desc="Validation")
#         for batch in val_loop:
#             input_ids = batch['input_ids'].to(elec_cfg.device)
#             attention_mask = batch['attention_mask'].to(elec_cfg.device)
#             labels = batch['label'].to(elec_cfg.device)
            
#             kwargs = {
#                 'input_ids': input_ids,
#                 'attention_mask': attention_mask,
#                 'labels': labels
#             }
#             if 'token_type_ids' in batch:
#                 kwargs['token_type_ids'] = batch['token_type_ids'].to(elec_cfg.device)
                
#             outputs = electra_model(**kwargs)
#             logits = outputs.logits  # Shape: [Batch_Size, 5]
            
#             val_loss += outputs.loss.item() * input_ids.size(0)

#             top1_preds = torch.argmax(logits, dim=1)
#             _, top3_indices = torch.topk(logits, k=3, dim=1)

#             all_top1_preds.append(top1_preds.cpu().numpy())
#             all_top3_preds.append(top3_indices.cpu().numpy())
#             all_targets.append(labels.cpu().numpy())
            
#     val_loss = val_loss / len(elec_val_loader.dataset)
#     all_top1_preds = np.concatenate(all_top1_preds, axis=0)
#     all_top3_preds = np.concatenate(all_top3_preds, axis=0)
#     all_targets = np.concatenate(all_targets, axis=0)
    
#     val_map3 = compute_map_at_3(all_top3_preds, all_targets)
#     val_accuracy = accuracy_score(all_targets, all_top1_preds)
#     val_f1 = f1_score(all_targets, all_top1_preds, average='macro')

#     print(f"Epoch {epoch+1}/{elec_cfg.epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
#           f"Val MAP@3: {val_map3:.4f} | Val Acc: {val_accuracy:.4f} | Val F1: {val_f1:.4f}")

#     wandb.log({
#         "electra/epoch": epoch + 1,
#         "electra/train_loss": train_loss,
#         "electra/val_loss": val_loss,
#         "electra/val_map3": val_map3,
#         "electra/val_accuracy": val_accuracy,
#         "electra/val_f1": val_f1
#     })

#     if val_map3 > best_map3:
#         best_map3 = val_map3
#         torch.save(electra_model.state_dict(), "best_electra_model.pt")
#         print(f"New electra best model saved with MAP@3: {best_map3:.4f}")

# wandb.finish()

# Inference

In [14]:
def repackage_pytorch_weights(source_dir, output_pt_path):
    with zipfile.ZipFile(output_pt_path, 'w', compression=zipfile.ZIP_STORED) as zf:
        parent_folder_name = os.path.basename(source_dir)
        
        for root, _, files in os.walk(source_dir):
            for file in files:
                file_path = os.path.join(root, file)
                rel_path = os.path.relpath(file_path, source_dir)
                archive_name = os.path.join(parent_folder_name, rel_path)
                zf.write(file_path, archive_name)

electra_dir = "/kaggle/input/datasets/adrijade4791/ensemble-best-weights/best_electra_model/best_electra_model"
bilstm_dir = "/kaggle/input/datasets/adrijade4791/ensemble-best-weights/best_bilstm_attn_model/best_bilstm_attn_model"

print("Reconstructing PyTorch archives without compression...")
repackage_pytorch_weights(electra_dir, "/tmp/best_electra.pt")
repackage_pytorch_weights(bilstm_dir, "/tmp/best_bilstm.pt")

print("Loading model state dicts...")
electra_model.load_state_dict(torch.load("/tmp/best_electra.pt"))
bilstm_model.load_state_dict(torch.load("/tmp/best_bilstm.pt"))


bilstm_model.eval()
electra_model.eval()

Reconstructing PyTorch archives without compression...
Loading model state dicts...


ElectraForMultipleChoice(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm

In [15]:
w_electra = 0.8
w_bilstm = 0.2

# Getting BiLSTM test probas
bilstm_test_probs = []
with torch.no_grad():
    for inputs in bilstm_test_loader:
        inputs = inputs.to(bilstm_cfg.device)
        outputs = bilstm_model(inputs)

        # softmax
        probs = torch.nn.functional.softmax(outputs, dim=1)
        bilstm_test_probs.append(probs.cpu())

bilstm_test_probs = torch.cat(bilstm_test_probs, dim=0)

# Getting Electra test probas
electra_test_probs = []
with torch.no_grad():
    for batch in elec_test_loader:
        input_ids = batch['input_ids'].to(elec_cfg.device)
        attention_mask = batch['attention_mask'].to(elec_cfg.device)
        
        kwargs = {
            'input_ids': input_ids,
            'attention_mask': attention_mask
        }
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(elec_cfg.device)
            
        outputs = electra_model(**kwargs)
        
        # softmax
        probs = torch.nn.functional.softmax(outputs.logits, dim=1)
        electra_test_probs.append(probs.cpu())

electra_test_probs = torch.cat(electra_test_probs, dim=0)

# Compute the weighted average of the probabilities
ensemble_probs = (w_bilstm * bilstm_test_probs) + (w_electra * electra_test_probs)

# Get the top 3 indices from the combined probabilities
_, top3_indices = torch.topk(ensemble_probs, k=3, dim=1)

idx_to_label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
submission_preds = []

for top_three in top3_indices.numpy():
    pred_str = " ".join([idx_to_label[i] for i in top_three])
    submission_preds.append(pred_str)

# Generate Submission

In [16]:
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'Prediction': submission_preds
})

submission_df.to_csv('submission.csv', index=False)
print("Submission saved to 'submission.csv'!")
print(submission_df.head())

Submission saved to 'submission.csv'!
   id Prediction
0   1      A C D
1   2      B D E
2   3      B E D
3   4      E C D
4   5      C B D
